In [14]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
import torch
from datasets import load_from_disk
from torch.utils.data import DataLoader

sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from lstm_data import LSTMDataset, infer_lstm_data_config, save_lstm_data_config
from lstm_model import BiLSTMForMultiLabelClassification
from lstm_trainer import train_lstm_model

In [15]:
dataset_full = load_from_disk('../data/processed/dataset_hf_processado')
datasets_split = dataset_full.train_test_split(test_size=0.2, seed=42)
dataset_train_hf = datasets_split['train']
dataset_test_hf = datasets_split['test']

with open('../data/processed/label_map.json', 'r') as f:
    id2label = json.load(f)

data_config = infer_lstm_data_config(dataset_full, pad_token_id=0)
save_lstm_data_config(data_config, '../data/processed/lstm_data_config.json')
num_labels = len(id2label)

train_dataset = LSTMDataset(dataset_train_hf)
test_dataset = LSTMDataset(dataset_test_hf)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [16]:
print(torch.cuda.is_available())
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cuda')
model = BiLSTMForMultiLabelClassification(
    vocab_size=data_config['vocab_size'],
    num_labels=num_labels,
    embedding_dim=200,
    hidden_size=256,
    num_layers=2,
    dropout=0.3,
    bidirectional=True,
    padding_idx=data_config['padding_idx'],
)

pos_weight = torch.load('../data/processed/pos_weights.pt', map_location='cpu', weights_only=True)

model, history = train_lstm_model(
    model=model,
    train_dataloader=train_loader,
    eval_dataloader=test_loader,
    device=device,
    learning_rate=2e-4,
    weight_decay=0.05,
    num_epochs=30,
    max_grad_norm=1.0,
    focal_gamma=2.0,
    pos_weight=pos_weight,
    early_stopping_patience=3,
)

print(f'Melhor modelo LSTM treinado em device={device}')

False


RuntimeError: No CUDA GPUs are available

In [ ]:
NOME_EXPERIMENTO = 'lstm_lr2e4_ep30_wd05_focal2_earlystop3_bs16'

PASTA_MODELO = Path('../models/lstm-cyber-threat')
PASTA_MODELO.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), PASTA_MODELO / 'pytorch_model.bin')

with (PASTA_MODELO / 'model_config.json').open('w', encoding='utf-8') as f:
    json.dump(
        {
            'model_type': 'bilstm_multilabel',
            'vocab_size': data_config['vocab_size'],
            'num_labels': num_labels,
            'embedding_dim': 200,
            'hidden_size': 256,
            'num_layers': 2,
            'dropout': 0.3,
            'bidirectional': True,
            'padding_idx': data_config['padding_idx'],
            'id2label': id2label,
        },
        f,
        ensure_ascii=True,
        indent=2,
    )

PASTA_HIST = Path('../data/processed/historic')
PASTA_HIST.mkdir(parents=True, exist_ok=True)
pd.DataFrame(history).to_csv(PASTA_HIST / f'{NOME_EXPERIMENTO}.csv', index=False)

print(f'Modelo salvo em: {PASTA_MODELO}')
print(f'Histórico salvo em: {PASTA_HIST / f"{NOME_EXPERIMENTO}.csv"}')

Modelo salvo em: ../models/lstm-cyber-threat
Histórico salvo em: ../data/processed/historic/lstm_lr2e4_ep30_wd05_focal2_earlystop3_bs16.csv
